# KG1 V207A H100 Official-Like ACC Gate Colab

Purpose: repair the score-facing gate before any new training.

This notebook:

- clones `FELIPEACASTRO/KG1-NVIDIA`;
- installs/repairs the V207A metric scripts inside `/content/kg1`;
- downloads the official train CSV mirror and builds a seed-42 stratified
  validation split for official-like ACC;
- evaluates the V194 production baseline with vLLM + LoRA;
- writes `predictions.csv`, `per_task.csv`, and a JSON report;
- runs a baseline self-compare smoke gate;
- does not train and never submits to Kaggle.


In [1]:
# CELL: mount Drive.
print('=== V207A DRIVE MOUNT START ===')
from google.colab import drive
drive.mount('/content/drive')
print('=== V207A DRIVE MOUNT END ===')


=== V207A DRIVE MOUNT START ===
Mounted at /content/drive
=== V207A DRIVE MOUNT END ===


In [2]:
# CELL: runtime configuration.
print('=== V207A CONFIG START ===')
import datetime
import hashlib
import json
import os
import pathlib
import subprocess
import sys

VERSION = 'V207A_H100_ACC_GATE_20260506'
REPO_URL = os.environ.get('KG1_REPO_URL', 'https://github.com/FELIPEACASTRO/KG1-NVIDIA.git')
REPO_BRANCH = os.environ.get('KG1_REPO_BRANCH', 'claude/competent-shamir')
ROOT = pathlib.Path('/content/kg1')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V207A')
OUT_ROOT = DRIVE_ROOT / 'output_v207a_acc_gate'
REPORT_DIR = OUT_ROOT / 'reports'
VAL_DIR = OUT_ROOT / 'validation'
BASELINE_ADAPTER = pathlib.Path('/content/drive/MyDrive/KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter')
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
MODEL_REVISION = 'cbd3fa9f933d55ef16a84236559f4ee2a0526848'
TRAIN_CSV_SHA256 = 'd204af160633b638448723a437aa51c0db70fd0b64ff92f6ad6f52e5ac6377fa'
TRAIN_CSV = VAL_DIR / 'official_train.csv'
VAL_CSV = VAL_DIR / 'official_train_seed42_stratified10_val.csv'
EVAL_LIMIT = int(os.environ.get('KG1_V207A_EVAL_LIMIT', '0'))  # 0 = full validation
ALLOW_KAGGLE_SUBMIT = False

for path in [DRIVE_ROOT, OUT_ROOT, REPORT_DIR, VAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('VERSION =', VERSION)
print('REPO_URL =', REPO_URL)
print('REPO_BRANCH =', REPO_BRANCH)
print('ROOT =', ROOT)
print('OUT_ROOT =', OUT_ROOT)
print('VAL_CSV =', VAL_CSV)
print('BASELINE_ADAPTER =', BASELINE_ADAPTER)
print('MODEL_NAME =', MODEL_NAME)
print('MODEL_REVISION =', MODEL_REVISION)
print('EVAL_LIMIT =', EVAL_LIMIT)
print('ALLOW_KAGGLE_SUBMIT =', ALLOW_KAGGLE_SUBMIT)
if ALLOW_KAGGLE_SUBMIT:
    raise RuntimeError('This notebook is submit-disabled by design.')
print('=== V207A CONFIG END ===')


=== V207A CONFIG START ===
VERSION = V207A_H100_ACC_GATE_20260506
REPO_URL = https://github.com/FELIPEACASTRO/KG1-NVIDIA.git
REPO_BRANCH = claude/competent-shamir
ROOT = /content/kg1
OUT_ROOT = /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate
VAL_CSV = /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_train_seed42_stratified10_val.csv
BASELINE_ADAPTER = /content/drive/MyDrive/KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter
MODEL_NAME = nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
MODEL_REVISION = cbd3fa9f933d55ef16a84236559f4ee2a0526848
EVAL_LIMIT = 0
ALLOW_KAGGLE_SUBMIT = False
=== V207A CONFIG END ===


In [3]:
# CELL: helper functions and dependency version logs.
print('=== V207A HELPERS START ===')
import importlib
import json
import pathlib
import subprocess
import sys
import time

def run_cmd(cmd, cwd=None, env=None, log_path=None, check=True):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    if log_path:
        log_path = pathlib.Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        handle = log_path.open('w', encoding='utf-8')
    else:
        handle = None
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        if handle:
            handle.write(line)
    rc = proc.wait()
    if handle:
        handle.close()
    print('returncode =', rc)
    if check and rc != 0:
        raise RuntimeError(f'Command failed with rc={rc}: {cmd}')
    return rc

def ensure_import(import_name, pip_spec=None):
    try:
        mod = importlib.import_module(import_name)
        print(import_name, 'version=', getattr(mod, '__version__', 'unknown'))
        return mod
    except Exception as exc:
        print(import_name, 'missing:', repr(exc))
        if not pip_spec:
            raise
        run_cmd([sys.executable, '-m', 'pip', 'install', '-q', pip_spec])
        mod = importlib.import_module(import_name)
        print(import_name, 'version=', getattr(mod, '__version__', 'unknown'))
        return mod

print('python =', sys.version)
print('=== V207A HELPERS END ===')


=== V207A HELPERS START ===
python = 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
=== V207A HELPERS END ===


In [6]:
# CELL: install runtime dependencies with explicit logs.
print('=== V207A DEPENDENCY CHECK START ===')

import importlib
import pathlib
import subprocess
import sys

def run_cmd(cmd, cwd=None, env=None, log_path=None, check=True):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    print('returncode =', rc)
    if check and rc != 0:
        raise RuntimeError(f'Command failed with rc={rc}: {cmd}')
    return rc

def ensure_import(import_name, pip_spec=None):
    try:
        mod = importlib.import_module(import_name)
        print(import_name, 'version=', getattr(mod, '__version__', 'unknown'))
        return mod
    except Exception as exc:
        print(import_name, 'missing/import failed:', repr(exc))
        if not pip_spec:
            raise
        run_cmd([sys.executable, '-m', 'pip', 'install', '-q', pip_spec])
        mod = importlib.import_module(import_name)
        print(import_name, 'version=', getattr(mod, '__version__', 'unknown'))
        return mod

def fresh_python_import_check(import_name):
    code = (
        "import importlib, torch; "
        f"m=importlib.import_module('{import_name}'); "
        "print('fresh_python_import_ok', m.__name__, getattr(m, '__version__', 'unknown')); "
        "print('fresh_python_torch', torch.__version__, getattr(torch.version, 'cuda', 'unknown'))"
    )
    run_cmd([sys.executable, '-c', code])

ensure_import('pandas', 'pandas')
ensure_import('huggingface_hub', 'huggingface_hub')
ensure_import('transformers', 'transformers')
ensure_import('peft', 'peft')

torch = ensure_import('torch')
print('torch_cuda_available =', torch.cuda.is_available())
print('torch_cuda_device_count =', torch.cuda.device_count() if torch.cuda.is_available() else 0)
if torch.cuda.is_available():
    print('torch_cuda_device_name =', torch.cuda.get_device_name(0))
    print('torch_cuda_version =', getattr(torch.version, 'cuda', 'unknown'))

try:
    ensure_import('vllm')
except Exception as exc:
    print('vLLM unavailable in current kernel before install:', repr(exc))
    print('Installing vLLM with CUDA 12.8 PyTorch wheel index.')
    print('Validation will run in a fresh Python process to avoid mixed torch/vLLM modules.')
    run_cmd([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--extra-index-url',
        'https://download.pytorch.org/whl/cu128',
        'vllm',
    ])
    fresh_python_import_check('vllm')
else:
    fresh_python_import_check('vllm')

print('=== V207A DEPENDENCY CHECK END ===')


=== V207A DEPENDENCY CHECK START ===
pandas version= 2.2.2
huggingface_hub version= 1.11.0
transformers version= 5.0.0
peft version= 0.19.1
torch version= 2.10.0+cu128
torch_cuda_available = True
torch_cuda_device_count = 1
torch_cuda_device_name = NVIDIA A100-SXM4-80GB
torch_cuda_version = 12.8
vllm missing/import failed: ImportError("cannot import name 'XPU_KERNEL_FORMAT' from 'torch._inductor.utils' (/usr/local/lib/python3.12/dist-packages/torch/_inductor/utils.py)")
vLLM unavailable in current kernel before install: ImportError("cannot import name 'XPU_KERNEL_FORMAT' from 'torch._inductor.utils' (/usr/local/lib/python3.12/dist-packages/torch/_inductor/utils.py)")
Installing vLLM with CUDA 12.8 PyTorch wheel index.
Validation will run in a fresh Python process to avoid mixed torch/vLLM modules.
+ /usr/bin/python3 -m pip install -q --extra-index-url https://download.pytorch.org/whl/cu128 vllm
returncode = 0
+ /usr/bin/python3 -c import importlib, torch; m=importlib.import_module('vll

In [7]:
# CELL: clone or update repository.
print('=== V207A REPO SETUP START ===')
if ROOT.exists():
    print('Repo exists; updating metadata only:', ROOT)
    run_cmd(['git', 'status', '--short'], cwd=ROOT, check=False)
else:
    run_cmd(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(ROOT)])
run_cmd(['git', 'rev-parse', '--show-toplevel'], cwd=ROOT, check=False)
run_cmd(['git', 'rev-parse', 'HEAD'], cwd=ROOT, check=False)
print('=== V207A REPO SETUP END ===')


=== V207A REPO SETUP START ===
+ git clone --branch claude/competent-shamir --depth 1 https://github.com/FELIPEACASTRO/KG1-NVIDIA.git /content/kg1
Cloning into '/content/kg1'...
returncode = 0
+ git rev-parse --show-toplevel
/content/kg1
returncode = 0
+ git rev-parse HEAD
bd85a6548cd98d52403716b3cad1ab69617c8568
returncode = 0
=== V207A REPO SETUP END ===


In [9]:
# CELL: install/repair V207A metric scripts inside the cloned repo.
print('=== V207A SCRIPT BOOTSTRAP START ===')
import json, pathlib, py_compile
ROOT = pathlib.Path('/content/kg1')
FILES = json.loads("{\n  \"src/__init__.py\": \"\\\"\\\"\\\"KG1 shared Python utilities.\\\"\\\"\\\"\\n\\n\",\n  \"src/competition_utils.py\": \"\\\"\\\"\\\"Shared metric utilities for the NVIDIA Nemotron reasoning challenge.\\n\\nThe answer extraction and verification functions intentionally mirror the\\npublic Kaggle metric path used by the Jiazhuang/Xduan local-CV notebooks:\\nextract the last boxed answer first, then fall back to final-answer phrases,\\nthen the last number, then the last non-empty line.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport re\\nimport unicodedata\\nfrom pathlib import Path\\nfrom typing import Any\\n\\n\\nREPO_ROOT = Path(__file__).resolve().parent.parent\\nDEFAULT_DATA_DIR = REPO_ROOT / \\\"data\\\"\\n\\nMODEL_NAME = \\\"nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16\\\"\\nMODEL_REVISION = \\\"cbd3fa9f933d55ef16a84236559f4ee2a0526848\\\"\\n\\nOFFICIAL_INFERENCE_CONFIG: dict[str, Any] = {\\n    \\\"model_name\\\": MODEL_NAME,\\n    \\\"model_revision\\\": MODEL_REVISION,\\n    \\\"max_lora_rank\\\": 32,\\n    \\\"max_tokens\\\": 7680,\\n    \\\"temperature\\\": 0.0,\\n    \\\"top_p\\\": 1.0,\\n    \\\"max_model_len\\\": 8192,\\n    \\\"max_num_seqs\\\": 64,\\n    \\\"gpu_memory_utilization\\\": 0.85,\\n    \\\"enable_prefix_caching\\\": True,\\n    \\\"enable_chunked_prefill\\\": True,\\n    \\\"trust_remote_code\\\": True,\\n    \\\"dtype\\\": \\\"auto\\\",\\n}\\n\\nPROMPT_SUFFIX = (\\n    \\\"\\\\nPlease put your final answer inside `\\\\\\\\boxed{}`. \\\"\\n    \\\"For example: `\\\\\\\\boxed{your answer}`\\\"\\n)\\n\\nFAMILIES = (\\n    \\\"gravity_constant\\\",\\n    \\\"unit_conversion\\\",\\n    \\\"numeral_system\\\",\\n    \\\"text_encryption\\\",\\n    \\\"bit_manipulation\\\",\\n    \\\"equation_transform\\\",\\n)\\n\\nFAMILY_ALIASES = {\\n    \\\"gravity\\\": \\\"gravity_constant\\\",\\n    \\\"grav\\\": \\\"gravity_constant\\\",\\n    \\\"gravity_constant\\\": \\\"gravity_constant\\\",\\n    \\\"unit\\\": \\\"unit_conversion\\\",\\n    \\\"units\\\": \\\"unit_conversion\\\",\\n    \\\"unit_conversion\\\": \\\"unit_conversion\\\",\\n    \\\"numeral\\\": \\\"numeral_system\\\",\\n    \\\"roman\\\": \\\"numeral_system\\\",\\n    \\\"roman_numeral\\\": \\\"numeral_system\\\",\\n    \\\"number_system\\\": \\\"numeral_system\\\",\\n    \\\"numeral_system\\\": \\\"numeral_system\\\",\\n    \\\"cipher\\\": \\\"text_encryption\\\",\\n    \\\"encryption\\\": \\\"text_encryption\\\",\\n    \\\"text\\\": \\\"text_encryption\\\",\\n    \\\"text_cipher\\\": \\\"text_encryption\\\",\\n    \\\"text_encryption\\\": \\\"text_encryption\\\",\\n    \\\"bit\\\": \\\"bit_manipulation\\\",\\n    \\\"bits\\\": \\\"bit_manipulation\\\",\\n    \\\"bit_manipulation\\\": \\\"bit_manipulation\\\",\\n    \\\"eq\\\": \\\"equation_transform\\\",\\n    \\\"equation\\\": \\\"equation_transform\\\",\\n    \\\"equation_rules\\\": \\\"equation_transform\\\",\\n    \\\"symbol_transform\\\": \\\"equation_transform\\\",\\n    \\\"equation_symbolic\\\": \\\"equation_transform\\\",\\n    \\\"equation_numeric\\\": \\\"equation_transform\\\",\\n    \\\"equation_numeric_deduce\\\": \\\"equation_transform\\\",\\n    \\\"equation_numeric_guess\\\": \\\"equation_transform\\\",\\n    \\\"cryptarithm_deduce\\\": \\\"equation_transform\\\",\\n    \\\"cryptarithm_guess\\\": \\\"equation_transform\\\",\\n    \\\"equation_transform\\\": \\\"equation_transform\\\",\\n}\\n\\n\\ndef _normalize_key(value: object) -> str:\\n    text = unicodedata.normalize(\\\"NFKC\\\", str(value or \\\"\\\")).strip().lower()\\n    return re.sub(r\\\"[\\\\s\\\\-]+\\\", \\\"_\\\", text)\\n\\n\\ndef canonical_family(value: object) -> str:\\n    key = _normalize_key(value)\\n    return FAMILY_ALIASES.get(key, key or \\\"unknown\\\")\\n\\n\\ndef classify_puzzle(prompt: str) -> str:\\n    low = str(prompt or \\\"\\\").lower()\\n    if \\\"bit manipulation\\\" in low or \\\"8-bit binary\\\" in low:\\n        return \\\"bit_manipulation\\\"\\n    if \\\"encryption\\\" in low or \\\"decrypt the following text\\\" in low or \\\"cipher\\\" in low:\\n        return \\\"text_encryption\\\"\\n    if \\\"numeral system\\\" in low or \\\"converted into a different numeral\\\" in low:\\n        return \\\"numeral_system\\\"\\n    if \\\"gravitational\\\" in low or \\\"gravity\\\" in low:\\n        return \\\"gravity_constant\\\"\\n    if \\\"transformation rule\\\" in low or \\\"transformation rules\\\" in low:\\n        return \\\"equation_transform\\\"\\n    if \\\"unit conversion\\\" in low or \\\"measurement\\\" in low:\\n        return \\\"unit_conversion\\\"\\n    return \\\"unknown\\\"\\n\\n\\ndef extract_boxed_answers(text: str | None) -> list[str]:\\n    if text is None:\\n        return []\\n    return re.findall(r\\\"\\\\\\\\boxed\\\\{([^}]*)(?:\\\\}|$)\\\", str(text))\\n\\n\\ndef extract_final_answer(text: str | None) -> str:\\n    \\\"\\\"\\\"Extract the final answer with the public Kaggle fallback order.\\\"\\\"\\\"\\n\\n    if text is None:\\n        return \\\"NOT_FOUND\\\"\\n    value = str(text)\\n\\n    matches = extract_boxed_answers(value)\\n    if matches:\\n        non_empty = [match.strip() for match in matches if match.strip()]\\n        if non_empty:\\n            return non_empty[-1]\\n        return matches[-1].strip()\\n\\n    patterns = [\\n        r\\\"The final answer is:\\\\s*([^\\\\n]+)\\\",\\n        r\\\"Final answer is:\\\\s*([^\\\\n]+)\\\",\\n        r\\\"Final answer\\\\s*[:\uff1a]\\\\s*([^\\\\n]+)\\\",\\n        r\\\"final answer\\\\s*[:\uff1a]\\\\s*([^\\\\n]+)\\\",\\n    ]\\n    for pattern in patterns:\\n        matches = re.findall(pattern, value, re.IGNORECASE)\\n        if matches:\\n            return matches[-1].strip()\\n\\n    matches = re.findall(r\\\"-?\\\\d+(?:\\\\.\\\\d+)?\\\", value)\\n    if matches:\\n        return matches[-1]\\n\\n    lines = [line.strip() for line in value.splitlines() if line.strip()]\\n    return lines[-1] if lines else \\\"NOT_FOUND\\\"\\n\\n\\ndef verify_answer(stored_answer: object, predicted: object) -> bool:\\n    \\\"\\\"\\\"Verify a prediction with the public Kaggle metric behavior.\\\"\\\"\\\"\\n\\n    expected = str(stored_answer).strip()\\n    observed = str(predicted).strip()\\n    if re.fullmatch(r\\\"[01]+\\\", expected):\\n        return observed.lower() == expected.lower()\\n    try:\\n        return math.isclose(float(expected), float(observed), rel_tol=1e-2, abs_tol=1e-5)\\n    except Exception:\\n        return observed.lower() == expected.lower()\\n\\n\\ndef canonical_answer(value: object) -> str:\\n    if value is None:\\n        return \\\"\\\"\\n    text = unicodedata.normalize(\\\"NFKC\\\", str(value))\\n    return re.sub(r\\\"\\\\s+\\\", \\\" \\\", text).strip()\\n\\n\\ndef escape_boxed_answer(value: object) -> str:\\n    return str(value).replace(\\\"\\\\\\\\\\\", \\\"\\\\\\\\\\\\\\\\\\\").replace(\\\"{\\\", \\\"\\\\\\\\{\\\").replace(\\\"}\\\", \\\"\\\\\\\\}\\\")\\n\\n\\ndef unescape_latex_braces(value: object) -> str:\\n    return str(value).replace(\\\"\\\\\\\\{\\\", \\\"{\\\").replace(\\\"\\\\\\\\}\\\", \\\"}\\\").replace(\\\"\\\\\\\\\\\\\\\\\\\", \\\"\\\\\\\\\\\")\\n\\n\\ndef canonical_boxed_payload(value: object) -> str:\\n    return canonical_answer(unescape_latex_braces(value))\\n\\n\\ndef parse_finite_number(value: object) -> float | None:\\n    text = canonical_answer(value).replace(\\\",\\\", \\\"\\\")\\n    if not text:\\n        return None\\n    try:\\n        number = float(text)\\n    except ValueError:\\n        return None\\n    return number if math.isfinite(number) else None\\n\\n\\ndef answers_equivalent(\\n    expected: object,\\n    observed: object,\\n    *,\\n    rel_tol: float = 1e-2,\\n    abs_tol: float = 1e-5,\\n    observed_is_boxed_payload: bool = False,\\n) -> bool:\\n    expected_text = canonical_answer(expected)\\n    observed_text = canonical_boxed_payload(observed) if observed_is_boxed_payload else canonical_answer(observed)\\n    expected_number = parse_finite_number(expected_text)\\n    observed_number = parse_finite_number(observed_text)\\n    if expected_number is not None and observed_number is not None:\\n        return math.isclose(expected_number, observed_number, rel_tol=rel_tol, abs_tol=abs_tol)\\n    return expected_text.lower() == observed_text.lower()\\n\\n\\ndef box_answer(value: object) -> str:\\n    return f\\\"\\\\\\\\boxed{{{escape_boxed_answer(value)}}}\\\"\\n\\n\",\n  \"scripts/evaluate_lora_adapter.py\": \"#!/usr/bin/env python3\\n\\\"\\\"\\\"Official-like vLLM evaluator for Nemotron LoRA adapters.\\n\\nThis script is intentionally evaluation-only. It does not train, package, or\\nsubmit. It generates answers with the same scoring-facing settings used by the\\npublic local-CV notebooks: LoRA enabled, max rank 32, 8192 context, 7680 output\\ntokens, deterministic sampling, boxed-answer extraction, and per-family ACC.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport os\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport pandas as pd\\n\\nROOT = Path(__file__).resolve().parents[1]\\nif str(ROOT) not in sys.path:\\n    sys.path.insert(0, str(ROOT))\\n\\nfrom src.competition_utils import (  # noqa: E402\\n    MODEL_NAME,\\n    OFFICIAL_INFERENCE_CONFIG,\\n    PROMPT_SUFFIX,\\n    classify_puzzle,\\n    extract_final_answer,\\n    verify_answer,\\n)\\n\\n\\ndef utc_now() -> str:\\n    return datetime.now(timezone.utc).isoformat()\\n\\n\\ndef parse_seeds(raw: str | int | None) -> list[int]:\\n    if raw is None or raw == \\\"\\\":\\n        return [42]\\n    if isinstance(raw, int):\\n        return [raw]\\n    seeds: list[int] = []\\n    for chunk in str(raw).replace(\\\";\\\", \\\",\\\").split(\\\",\\\"):\\n        chunk = chunk.strip()\\n        if chunk:\\n            seeds.append(int(chunk))\\n    return seeds or [42]\\n\\n\\ndef resolve_base_model_path(base_model_path: str = \\\"\\\") -> str:\\n    \\\"\\\"\\\"Resolve the base model path for Colab, Kaggle, or local H100 runs.\\\"\\\"\\\"\\n\\n    if base_model_path:\\n        return base_model_path\\n    env_path = os.environ.get(\\\"KG1_BASE_MODEL_PATH\\\") or os.environ.get(\\\"BASE_MODEL_PATH\\\")\\n    if env_path:\\n        return env_path\\n\\n    kaggle_candidates = [\\n        \\\"/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1\\\",\\n        \\\"/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default\\\",\\n        \\\"/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1\\\",\\n    ]\\n    for candidate in kaggle_candidates:\\n        if Path(candidate).exists():\\n            return candidate\\n    return MODEL_NAME\\n\\n\\ndef resolve_model_revision(base_model_path: str, config: dict[str, Any]) -> str | None:\\n    \\\"\\\"\\\"Use the pinned HF revision for repo IDs, but not for local model paths.\\\"\\\"\\\"\\n\\n    revision = str(config.get(\\\"model_revision\\\") or \\\"\\\").strip()\\n    if not revision:\\n        return None\\n    model_text = str(base_model_path)\\n    if Path(model_text).exists() or os.path.isabs(model_text):\\n        return None\\n    return revision\\n\\n\\ndef row_id_column(frame: pd.DataFrame) -> str:\\n    for candidate in (\\\"id\\\", \\\"row_id\\\"):\\n        if candidate in frame.columns:\\n            return candidate\\n    return str(frame.columns.to_list()[0])\\n\\n\\ndef normalize_questions(solution: pd.DataFrame, questions: pd.DataFrame, limit: int = 0) -> pd.DataFrame:\\n    solution = solution.copy()\\n    questions = questions.copy()\\n    sol_id = row_id_column(solution)\\n    q_id = row_id_column(questions)\\n    if sol_id != \\\"id\\\":\\n        solution = solution.rename(columns={sol_id: \\\"id\\\"})\\n    if q_id != \\\"id\\\":\\n        questions = questions.rename(columns={q_id: \\\"id\\\"})\\n    solution[\\\"id\\\"] = solution[\\\"id\\\"].astype(str)\\n    questions[\\\"id\\\"] = questions[\\\"id\\\"].astype(str)\\n    if \\\"prompt\\\" not in questions.columns:\\n        if \\\"prompt\\\" not in solution.columns:\\n            raise ValueError(\\\"questions or solution must contain a prompt column\\\")\\n        questions = solution[[\\\"id\\\", \\\"prompt\\\"]].copy()\\n    ordered = solution[[\\\"id\\\"]].merge(questions, on=\\\"id\\\", how=\\\"left\\\", validate=\\\"one_to_one\\\")\\n    missing_prompt = ordered[\\\"prompt\\\"].isna().sum()\\n    if missing_prompt:\\n        raise ValueError(f\\\"questions missing prompts for {missing_prompt} solution rows\\\")\\n    if limit > 0:\\n        ordered = ordered.head(limit).copy()\\n    return ordered\\n\\n\\ndef validate_adapter_dir(adapter_dir: str | Path) -> Path:\\n    path = Path(adapter_dir)\\n    if not path.exists():\\n        raise FileNotFoundError(f\\\"adapter path does not exist: {path}\\\")\\n    if path.is_file() and path.suffix == \\\".zip\\\":\\n        raise ValueError(\\\"adapter zip must be extracted before vLLM evaluation\\\")\\n    config = path / \\\"adapter_config.json\\\"\\n    if not config.exists():\\n        raise FileNotFoundError(f\\\"missing adapter_config.json: {config}\\\")\\n    model_files = list(path.glob(\\\"adapter_model.safetensors\\\")) + list(path.glob(\\\"adapter_model.bin\\\"))\\n    if not model_files:\\n        raise FileNotFoundError(f\\\"missing adapter_model.safetensors or adapter_model.bin in {path}\\\")\\n    return path\\n\\n\\ndef render_prompts(tokenizer: Any, questions: pd.DataFrame) -> list[str]:\\n    prompts: list[str] = []\\n    for row in questions.itertuples(index=False):\\n        user_content = str(getattr(row, \\\"prompt\\\")) + PROMPT_SUFFIX\\n        try:\\n            prompt = tokenizer.apply_chat_template(\\n                [{\\\"role\\\": \\\"user\\\", \\\"content\\\": user_content}],\\n                tokenize=False,\\n                add_generation_prompt=True,\\n                enable_thinking=True,\\n            )\\n        except Exception:\\n            prompt = user_content\\n        prompts.append(prompt)\\n    return prompts\\n\\n\\ndef _sampling_params(config: dict[str, Any], seed: int):\\n    from vllm import SamplingParams\\n\\n    kwargs = {\\n        \\\"temperature\\\": float(config.get(\\\"temperature\\\", 0.0)),\\n        \\\"top_p\\\": float(config.get(\\\"top_p\\\", 1.0)),\\n        \\\"max_tokens\\\": int(config.get(\\\"max_tokens\\\", 7680)),\\n    }\\n    try:\\n        return SamplingParams(**kwargs, seed=int(seed))\\n    except TypeError:\\n        return SamplingParams(**kwargs)\\n\\n\\ndef evaluate_adapter(\\n    solution: pd.DataFrame,\\n    questions: pd.DataFrame,\\n    *,\\n    lora_path: str,\\n    base_model_path: str,\\n    config: dict[str, Any] | None = None,\\n    seed: int = 42,\\n) -> tuple[dict[str, Any], pd.DataFrame]:\\n    \\\"\\\"\\\"Run vLLM adapter inference and return a summary plus row predictions.\\\"\\\"\\\"\\n\\n    config = {**OFFICIAL_INFERENCE_CONFIG, **(config or {})}\\n    adapter_dir = validate_adapter_dir(lora_path)\\n    questions = normalize_questions(solution, questions, limit=0)\\n    solution = solution.copy()\\n    id_col = row_id_column(solution)\\n    if id_col != \\\"id\\\":\\n        solution = solution.rename(columns={id_col: \\\"id\\\"})\\n    solution[\\\"id\\\"] = solution[\\\"id\\\"].astype(str)\\n\\n    print(\\\"========================================================================\\\")\\n    print(\\\"KG1 official-like adapter evaluation\\\")\\n    print(\\\"========================================================================\\\")\\n    print(\\\"generated_at_utc =\\\", utc_now())\\n    print(\\\"base_model_path =\\\", base_model_path)\\n    print(\\\"adapter_dir =\\\", adapter_dir)\\n    print(\\\"rows =\\\", len(questions))\\n    print(\\\"seed =\\\", seed)\\n    print(\\\"config =\\\", json.dumps(config, indent=2, sort_keys=True))\\n\\n    from vllm import LLM\\n    from vllm.lora.request import LoRARequest\\n\\n    llm_kwargs = {\\n        \\\"model\\\": str(base_model_path),\\n        \\\"tensor_parallel_size\\\": int(config.get(\\\"tensor_parallel_size\\\", 1)),\\n        \\\"max_num_seqs\\\": int(config.get(\\\"max_num_seqs\\\", 64)),\\n        \\\"gpu_memory_utilization\\\": float(config.get(\\\"gpu_memory_utilization\\\", 0.85)),\\n        \\\"dtype\\\": config.get(\\\"dtype\\\", \\\"auto\\\"),\\n        \\\"max_model_len\\\": int(config.get(\\\"max_model_len\\\", 8192)),\\n        \\\"trust_remote_code\\\": bool(config.get(\\\"trust_remote_code\\\", True)),\\n        \\\"enable_lora\\\": True,\\n        \\\"max_lora_rank\\\": int(config.get(\\\"max_lora_rank\\\", 32)),\\n        \\\"enable_prefix_caching\\\": bool(config.get(\\\"enable_prefix_caching\\\", True)),\\n        \\\"enable_chunked_prefill\\\": bool(config.get(\\\"enable_chunked_prefill\\\", True)),\\n    }\\n    model_revision = resolve_model_revision(str(base_model_path), config)\\n    if model_revision:\\n        llm_kwargs[\\\"revision\\\"] = model_revision\\n        llm_kwargs[\\\"tokenizer_revision\\\"] = model_revision\\n    print(\\\"llm_revision =\\\", llm_kwargs.get(\\\"revision\\\", \\\"local_path_or_default\\\"))\\n    if config.get(\\\"enforce_eager\\\") is not None:\\n        llm_kwargs[\\\"enforce_eager\\\"] = bool(config[\\\"enforce_eager\\\"])\\n\\n    start = time.time()\\n    llm = LLM(**llm_kwargs)\\n    tokenizer = llm.get_tokenizer()\\n    print(f\\\"vLLM loaded in {time.time() - start:.1f}s\\\")\\n\\n    rendered = render_prompts(tokenizer, questions)\\n    sampling_params = _sampling_params(config, seed)\\n    lora_request = LoRARequest(\\\"adapter\\\", 1, str(adapter_dir))\\n\\n    if rendered:\\n        warmup_n = min(4, len(rendered))\\n        print(f\\\"warmup_rows = {warmup_n}\\\")\\n        warmup_start = time.time()\\n        _ = llm.generate(rendered[:warmup_n], sampling_params=sampling_params, lora_request=lora_request)\\n        print(f\\\"warmup_elapsed_s = {time.time() - warmup_start:.1f}\\\")\\n\\n    gen_start = time.time()\\n    outputs = llm.generate(rendered, sampling_params=sampling_params, lora_request=lora_request)\\n    gen_elapsed = time.time() - gen_start\\n    print(f\\\"generation_elapsed_s = {gen_elapsed:.1f}\\\")\\n\\n    rows: list[dict[str, Any]] = []\\n    for row, output in zip(questions.itertuples(index=False), outputs):\\n        completion = output.outputs[0]\\n        raw_output = completion.text\\n        prediction = extract_final_answer(raw_output)\\n        row_id = str(getattr(row, \\\"id\\\"))\\n        prompt = str(getattr(row, \\\"prompt\\\"))\\n        rows.append(\\n            {\\n                \\\"id\\\": row_id,\\n                \\\"prompt\\\": prompt,\\n                \\\"raw_output\\\": raw_output,\\n                \\\"prediction\\\": prediction,\\n                \\\"prompt_tokens\\\": len(getattr(output, \\\"prompt_token_ids\\\", []) or []),\\n                \\\"completion_tokens\\\": len(getattr(completion, \\\"token_ids\\\", []) or []),\\n                \\\"finish_reason\\\": completion.finish_reason or \\\"\\\",\\n                \\\"type\\\": classify_puzzle(prompt),\\n            }\\n        )\\n\\n    pred = pd.DataFrame(rows)\\n    merged = solution.merge(pred, on=\\\"id\\\", how=\\\"left\\\", validate=\\\"one_to_one\\\")\\n    if \\\"answer\\\" in merged.columns:\\n        merged[\\\"correct\\\"] = merged.apply(lambda r: verify_answer(r[\\\"answer\\\"], r[\\\"prediction\\\"]), axis=1)\\n    else:\\n        merged[\\\"correct\\\"] = False\\n    if \\\"type\\\" not in merged.columns:\\n        merged[\\\"type\\\"] = merged[\\\"prompt\\\"].map(classify_puzzle)\\n    merged[\\\"truncated\\\"] = merged[\\\"finish_reason\\\"].fillna(\\\"\\\").astype(str).eq(\\\"length\\\")\\n\\n    total_tokens = int(merged[\\\"completion_tokens\\\"].fillna(0).sum())\\n    summary = {\\n        \\\"generated_at_utc\\\": utc_now(),\\n        \\\"base_model_path\\\": str(base_model_path),\\n        \\\"adapter_dir\\\": str(adapter_dir),\\n        \\\"rows\\\": int(len(merged)),\\n        \\\"correct\\\": int(merged[\\\"correct\\\"].sum()),\\n        \\\"accuracy\\\": float(merged[\\\"correct\\\"].mean()) if len(merged) else 0.0,\\n        \\\"truncated\\\": int(merged[\\\"truncated\\\"].sum()),\\n        \\\"truncation_rate\\\": float(merged[\\\"truncated\\\"].mean()) if len(merged) else 0.0,\\n        \\\"completion_tokens\\\": total_tokens,\\n        \\\"generation_elapsed_s\\\": gen_elapsed,\\n        \\\"tokens_per_second\\\": float(total_tokens / gen_elapsed) if gen_elapsed > 0 else 0.0,\\n        \\\"seed\\\": int(seed),\\n        \\\"config\\\": config,\\n    }\\n    print(\\\"summary =\\\", json.dumps(summary, indent=2, sort_keys=True))\\n    return summary, merged\\n\\n\\ndef summarize_per_task(frame: pd.DataFrame) -> pd.DataFrame:\\n    rows: list[dict[str, Any]] = []\\n    grouped = frame.groupby(\\\"type\\\", dropna=False)\\n    for family, group in grouped:\\n        total = int(len(group))\\n        correct = int(group[\\\"correct\\\"].sum())\\n        truncated = int(group[\\\"truncated\\\"].sum()) if \\\"truncated\\\" in group else 0\\n        rows.append(\\n            {\\n                \\\"task_type\\\": str(family),\\n                \\\"total\\\": total,\\n                \\\"correct\\\": correct,\\n                \\\"accuracy\\\": correct / total if total else 0.0,\\n                \\\"truncated\\\": truncated,\\n                \\\"truncation_rate\\\": truncated / total if total else 0.0,\\n            }\\n        )\\n    total = int(len(frame))\\n    correct = int(frame[\\\"correct\\\"].sum()) if \\\"correct\\\" in frame else 0\\n    truncated = int(frame[\\\"truncated\\\"].sum()) if \\\"truncated\\\" in frame else 0\\n    rows.append(\\n        {\\n            \\\"task_type\\\": \\\"OVERALL\\\",\\n            \\\"total\\\": total,\\n            \\\"correct\\\": correct,\\n            \\\"accuracy\\\": correct / total if total else 0.0,\\n            \\\"truncated\\\": truncated,\\n            \\\"truncation_rate\\\": truncated / total if total else 0.0,\\n        }\\n    )\\n    return pd.DataFrame(rows)\\n\\n\\ndef main() -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\\"--solution-csv\\\", type=Path, required=True)\\n    parser.add_argument(\\\"--questions-csv\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--adapter\\\", type=Path, required=True)\\n    parser.add_argument(\\\"--base-model-path\\\", default=\\\"\\\")\\n    parser.add_argument(\\\"--label\\\", default=\\\"adapter\\\")\\n    parser.add_argument(\\\"--seed\\\", type=int, default=42)\\n    parser.add_argument(\\\"--limit\\\", type=int, default=0)\\n    parser.add_argument(\\\"--output-dir\\\", type=Path, required=True)\\n    args = parser.parse_args()\\n\\n    args.output_dir.mkdir(parents=True, exist_ok=True)\\n    solution = pd.read_csv(args.solution_csv)\\n    if args.limit > 0:\\n        solution = solution.head(args.limit).copy()\\n    questions = pd.read_csv(args.questions_csv or args.solution_csv)\\n    if args.limit > 0:\\n        ids = set(solution[row_id_column(solution)].astype(str))\\n        q_id = row_id_column(questions)\\n        questions = questions[questions[q_id].astype(str).isin(ids)].copy()\\n\\n    summary, predictions = evaluate_adapter(\\n        solution,\\n        questions,\\n        lora_path=str(args.adapter),\\n        base_model_path=resolve_base_model_path(args.base_model_path),\\n        config=OFFICIAL_INFERENCE_CONFIG,\\n        seed=args.seed,\\n    )\\n    label = args.label.replace(\\\"/\\\", \\\"_\\\").replace(\\\"\\\\\\\\\\\", \\\"_\\\")\\n    predictions_path = args.output_dir / f\\\"{label}_predictions.csv\\\"\\n    per_task_path = args.output_dir / f\\\"{label}_per_task.csv\\\"\\n    report_path = args.output_dir / f\\\"{label}_eval_report.json\\\"\\n    predictions.to_csv(predictions_path, index=False)\\n    summarize_per_task(predictions).to_csv(per_task_path, index=False)\\n    report = {\\n        **summary,\\n        \\\"label\\\": args.label,\\n        \\\"inputs\\\": {\\n            \\\"solution_csv\\\": str(args.solution_csv),\\n            \\\"questions_csv\\\": str(args.questions_csv or args.solution_csv),\\n            \\\"adapter\\\": str(args.adapter),\\n            \\\"limit\\\": args.limit,\\n        },\\n        \\\"outputs\\\": {\\n            \\\"predictions_csv\\\": str(predictions_path),\\n            \\\"per_task_csv\\\": str(per_task_path),\\n            \\\"report_json\\\": str(report_path),\\n        },\\n    }\\n    report_path.write_text(json.dumps(report, indent=2, sort_keys=True), encoding=\\\"utf-8\\\")\\n    print(\\\"predictions_csv =\\\", predictions_path)\\n    print(\\\"per_task_csv =\\\", per_task_path)\\n    print(\\\"report_json =\\\", report_path)\\n    return 0\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    raise SystemExit(main())\\n\",\n  \"scripts/solve_rate_gate.py\": \"#!/usr/bin/env python3\\n\\\"\\\"\\\"Solve-rate promotion gate for Nemotron LoRA candidates.\\n\\nThis is the score-facing gate: compare candidate vs baseline by decoded\\nanswers, official answer verification, and per-family regressions. It supports\\ntwo modes:\\n\\n1. CSV mode: compare existing prediction CSVs.\\n2. Adapter mode: run vLLM evaluation for baseline and candidate adapters.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport statistics\\nimport sys\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport pandas as pd\\n\\nROOT = Path(__file__).resolve().parents[1]\\nif str(ROOT) not in sys.path:\\n    sys.path.insert(0, str(ROOT))\\n\\nfrom scripts.evaluate_lora_adapter import evaluate_adapter, parse_seeds, resolve_base_model_path\\nfrom src.competition_utils import (\\n    OFFICIAL_INFERENCE_CONFIG,\\n    classify_puzzle,\\n    extract_boxed_answers,\\n    extract_final_answer,\\n    verify_answer,\\n)\\n\\n\\ndef utc_now() -> str:\\n    return datetime.now(timezone.utc).isoformat()\\n\\n\\ndef row_id_column(frame: pd.DataFrame) -> str:\\n    for candidate in (\\\"id\\\", \\\"row_id\\\"):\\n        if candidate in frame.columns:\\n            return candidate\\n    return str(frame.columns.to_list()[0])\\n\\n\\ndef family_for_row(row: pd.Series) -> str:\\n    for key in (\\\"type\\\", \\\"family\\\", \\\"task_family\\\"):\\n        value = row.get(key)\\n        if value not in (None, \\\"\\\"):\\n            return str(value)\\n    return classify_puzzle(str(row.get(\\\"prompt\\\", \\\"\\\")))\\n\\n\\ndef normalize_solution(solution_csv: Path, limit: int = 0) -> pd.DataFrame:\\n    solution = pd.read_csv(solution_csv)\\n    required = {\\\"prompt\\\", \\\"answer\\\"}\\n    missing = sorted(required - set(solution.columns))\\n    if missing:\\n        raise ValueError(f\\\"solution CSV missing required columns: {missing}\\\")\\n    id_col = row_id_column(solution)\\n    if id_col != \\\"id\\\":\\n        solution = solution.rename(columns={id_col: \\\"id\\\"})\\n    solution[\\\"id\\\"] = solution[\\\"id\\\"].astype(str)\\n    solution[\\\"family_gate\\\"] = solution.apply(family_for_row, axis=1)\\n    if limit > 0:\\n        solution = solution.head(limit).copy()\\n    return solution\\n\\n\\ndef predictions_from_csv(predictions_csv: Path, label: str) -> pd.DataFrame:\\n    predictions = pd.read_csv(predictions_csv)\\n    id_col = row_id_column(predictions)\\n    if id_col != \\\"id\\\":\\n        predictions = predictions.rename(columns={id_col: \\\"id\\\"})\\n    predictions[\\\"id\\\"] = predictions[\\\"id\\\"].astype(str)\\n\\n    if \\\"raw_output\\\" not in predictions.columns and \\\"prediction\\\" not in predictions.columns:\\n        raise ValueError(f\\\"{label} predictions need a 'prediction' or 'raw_output' column\\\")\\n    if \\\"raw_output\\\" not in predictions.columns:\\n        predictions[\\\"raw_output\\\"] = predictions[\\\"prediction\\\"].astype(str)\\n    if \\\"prediction\\\" not in predictions.columns:\\n        predictions[\\\"prediction\\\"] = predictions[\\\"raw_output\\\"].map(extract_final_answer)\\n    else:\\n        missing_prediction = predictions[\\\"prediction\\\"].isna() | (predictions[\\\"prediction\\\"].astype(str) == \\\"\\\")\\n        predictions.loc[missing_prediction, \\\"prediction\\\"] = predictions.loc[\\n            missing_prediction, \\\"raw_output\\\"\\n        ].map(extract_final_answer)\\n    return predictions[[\\\"id\\\", \\\"prediction\\\", \\\"raw_output\\\"]].copy()\\n\\n\\ndef score_predictions(solution: pd.DataFrame, predictions: pd.DataFrame, label: str, seed: int | None = None) -> pd.DataFrame:\\n    merged = solution.merge(predictions, on=\\\"id\\\", how=\\\"left\\\", validate=\\\"one_to_one\\\")\\n    merged[\\\"label\\\"] = label\\n    merged[\\\"seed\\\"] = seed if seed is not None else 0\\n    merged[\\\"prediction\\\"] = merged[\\\"prediction\\\"].fillna(\\\"NOT_FOUND\\\").astype(str)\\n    merged[\\\"raw_output\\\"] = merged[\\\"raw_output\\\"].fillna(\\\"\\\").astype(str)\\n    merged[\\\"final_answer\\\"] = merged.apply(\\n        lambda row: extract_final_answer(row[\\\"raw_output\\\"]) if row[\\\"raw_output\\\"] else row[\\\"prediction\\\"],\\n        axis=1,\\n    )\\n    merged[\\\"boxed_valid\\\"] = merged[\\\"raw_output\\\"].map(lambda value: len(extract_boxed_answers(value)) > 0)\\n    merged[\\\"correct\\\"] = merged.apply(\\n        lambda row: verify_answer(str(row[\\\"answer\\\"]), str(row[\\\"final_answer\\\"])),\\n        axis=1,\\n    )\\n    return merged\\n\\n\\ndef prediction_frames_from_adapters(\\n    solution: pd.DataFrame,\\n    questions: pd.DataFrame,\\n    *,\\n    baseline_adapter: Path,\\n    candidate_adapter: Path,\\n    base_model_path: str,\\n    seeds: list[int],\\n) -> tuple[pd.DataFrame, pd.DataFrame]:\\n    baseline_frames: list[pd.DataFrame] = []\\n    candidate_frames: list[pd.DataFrame] = []\\n    for seed in seeds:\\n        _, baseline_merged = evaluate_adapter(\\n            solution,\\n            questions,\\n            lora_path=str(baseline_adapter),\\n            base_model_path=base_model_path,\\n            config=OFFICIAL_INFERENCE_CONFIG,\\n            seed=seed,\\n        )\\n        _, candidate_merged = evaluate_adapter(\\n            solution,\\n            questions,\\n            lora_path=str(candidate_adapter),\\n            base_model_path=base_model_path,\\n            config=OFFICIAL_INFERENCE_CONFIG,\\n            seed=seed,\\n        )\\n        baseline_merged = baseline_merged.rename(columns={\\\"type\\\": \\\"family_gate\\\"})\\n        candidate_merged = candidate_merged.rename(columns={\\\"type\\\": \\\"family_gate\\\"})\\n        baseline_frames.append(score_predictions(solution, baseline_merged[[\\\"id\\\", \\\"prediction\\\", \\\"raw_output\\\"]], \\\"baseline\\\", seed))\\n        candidate_frames.append(score_predictions(solution, candidate_merged[[\\\"id\\\", \\\"prediction\\\", \\\"raw_output\\\"]], \\\"candidate\\\", seed))\\n    return pd.concat(baseline_frames, ignore_index=True), pd.concat(candidate_frames, ignore_index=True)\\n\\n\\ndef summarize(frame: pd.DataFrame) -> dict[str, Any]:\\n    by_family: dict[str, dict[str, Any]] = {}\\n    for family, group in frame.groupby(\\\"family_gate\\\"):\\n        by_family[str(family)] = {\\n            \\\"rows\\\": int(len(group)),\\n            \\\"correct\\\": int(group[\\\"correct\\\"].sum()),\\n            \\\"accuracy\\\": float(group[\\\"correct\\\"].mean()) if len(group) else 0.0,\\n            \\\"boxed_format_rate\\\": float(group[\\\"boxed_valid\\\"].mean()) if len(group) else 0.0,\\n        }\\n\\n    seed_summaries = []\\n    for seed, group in frame.groupby(\\\"seed\\\"):\\n        seed_summaries.append(\\n            {\\n                \\\"seed\\\": int(seed),\\n                \\\"rows\\\": int(len(group)),\\n                \\\"correct\\\": int(group[\\\"correct\\\"].sum()),\\n                \\\"accuracy\\\": float(group[\\\"correct\\\"].mean()) if len(group) else 0.0,\\n                \\\"boxed_format_rate\\\": float(group[\\\"boxed_valid\\\"].mean()) if len(group) else 0.0,\\n            }\\n        )\\n    accuracies = [item[\\\"accuracy\\\"] for item in seed_summaries]\\n    return {\\n        \\\"rows\\\": int(len(frame)),\\n        \\\"correct\\\": int(frame[\\\"correct\\\"].sum()),\\n        \\\"accuracy\\\": float(frame[\\\"correct\\\"].mean()) if len(frame) else 0.0,\\n        \\\"boxed_format_rate\\\": float(frame[\\\"boxed_valid\\\"].mean()) if len(frame) else 0.0,\\n        \\\"accuracy_min_seed\\\": float(min(accuracies)) if accuracies else 0.0,\\n        \\\"accuracy_mean_seed\\\": float(statistics.mean(accuracies)) if accuracies else 0.0,\\n        \\\"accuracy_max_seed\\\": float(max(accuracies)) if accuracies else 0.0,\\n        \\\"by_seed\\\": seed_summaries,\\n        \\\"by_family\\\": by_family,\\n    }\\n\\n\\ndef compare(\\n    baseline: pd.DataFrame,\\n    candidate: pd.DataFrame,\\n    *,\\n    family_regression_tolerance: float,\\n    min_net_gain: float,\\n    min_boxed_rate: float,\\n) -> tuple[bool, list[str], dict[str, Any]]:\\n    baseline_summary = summarize(baseline)\\n    candidate_summary = summarize(candidate)\\n    reasons: list[str] = []\\n\\n    baseline_accuracy = float(baseline_summary[\\\"accuracy\\\"])\\n    candidate_accuracy = float(candidate_summary[\\\"accuracy\\\"])\\n    net_gain = candidate_accuracy - baseline_accuracy\\n    if net_gain <= min_net_gain:\\n        reasons.append(f\\\"net_gain_not_above_threshold:{net_gain:.6f}<={min_net_gain:.6f}\\\")\\n\\n    if float(candidate_summary[\\\"boxed_format_rate\\\"]) < min_boxed_rate:\\n        reasons.append(\\n            \\\"candidate_boxed_rate_below_threshold:\\\"\\n            f\\\"{candidate_summary['boxed_format_rate']:.6f}<{min_boxed_rate:.6f}\\\"\\n        )\\n\\n    baseline_families = baseline_summary[\\\"by_family\\\"]\\n    candidate_families = candidate_summary[\\\"by_family\\\"]\\n    family_deltas: dict[str, dict[str, float]] = {}\\n    for family in sorted(set(baseline_families) | set(candidate_families)):\\n        b_acc = float(baseline_families.get(family, {}).get(\\\"accuracy\\\", 0.0))\\n        c_acc = float(candidate_families.get(family, {}).get(\\\"accuracy\\\", 0.0))\\n        delta = c_acc - b_acc\\n        family_deltas[family] = {\\n            \\\"baseline_accuracy\\\": b_acc,\\n            \\\"candidate_accuracy\\\": c_acc,\\n            \\\"delta\\\": delta,\\n        }\\n        if delta < -family_regression_tolerance:\\n            reasons.append(\\n                f\\\"family_regression:{family}:{c_acc:.6f}<{b_acc:.6f}-\\\"\\n                f\\\"{family_regression_tolerance:.6f}\\\"\\n            )\\n\\n    comparison = {\\n        \\\"baseline\\\": baseline_summary,\\n        \\\"candidate\\\": candidate_summary,\\n        \\\"net_gain\\\": net_gain,\\n        \\\"family_deltas\\\": family_deltas,\\n    }\\n    return len(reasons) == 0, reasons, comparison\\n\\n\\ndef write_failures(path: Path, baseline: pd.DataFrame, candidate: pd.DataFrame) -> None:\\n    cols = [\\\"id\\\", \\\"family_gate\\\", \\\"answer\\\", \\\"final_answer\\\", \\\"correct\\\"]\\n    b = baseline[cols].rename(columns={\\\"final_answer\\\": \\\"baseline_final_answer\\\", \\\"correct\\\": \\\"baseline_correct\\\"})\\n    c = candidate[cols].rename(columns={\\\"final_answer\\\": \\\"candidate_final_answer\\\", \\\"correct\\\": \\\"candidate_correct\\\"})\\n    merged = b.merge(c, on=[\\\"id\\\", \\\"family_gate\\\", \\\"answer\\\"], how=\\\"outer\\\")\\n    merged[\\\"regressed\\\"] = merged[\\\"baseline_correct\\\"].fillna(False) & ~merged[\\\"candidate_correct\\\"].fillna(False)\\n    merged[\\\"improved\\\"] = ~merged[\\\"baseline_correct\\\"].fillna(False) & merged[\\\"candidate_correct\\\"].fillna(False)\\n    merged.to_csv(path, index=False)\\n\\n\\ndef run_self_test(solution_csv: Path, output_dir: Path, limit: int) -> int:\\n    solution = normalize_solution(solution_csv, limit=limit or 20)\\n    baseline_predictions = pd.DataFrame(\\n        {\\n            \\\"id\\\": solution[\\\"id\\\"],\\n            \\\"prediction\\\": solution[\\\"answer\\\"].astype(str),\\n            \\\"raw_output\\\": \\\"\\\\\\\\boxed{\\\" + solution[\\\"answer\\\"].astype(str) + \\\"}\\\",\\n        }\\n    )\\n    candidate_predictions = baseline_predictions.copy()\\n    if len(candidate_predictions):\\n        candidate_predictions.loc[candidate_predictions.index[-1], \\\"prediction\\\"] = \\\"INTENTIONAL_WRONG\\\"\\n        candidate_predictions.loc[candidate_predictions.index[-1], \\\"raw_output\\\"] = \\\"\\\\\\\\boxed{INTENTIONAL_WRONG}\\\"\\n    baseline = score_predictions(solution, baseline_predictions, \\\"baseline\\\")\\n    candidate = score_predictions(solution, candidate_predictions, \\\"candidate\\\")\\n    approved, reasons, comparison = compare(\\n        baseline,\\n        candidate,\\n        family_regression_tolerance=0.0,\\n        min_net_gain=0.0,\\n        min_boxed_rate=1.0,\\n    )\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n    payload = {\\n        \\\"self_test\\\": True,\\n        \\\"approved\\\": approved,\\n        \\\"reasons\\\": reasons,\\n        \\\"comparison\\\": comparison,\\n    }\\n    (output_dir / \\\"self_test_report.json\\\").write_text(json.dumps(payload, indent=2), encoding=\\\"utf-8\\\")\\n    write_failures(output_dir / \\\"self_test_row_deltas.csv\\\", baseline, candidate)\\n    print(json.dumps(payload, indent=2))\\n    return 0 if not approved and reasons else 2\\n\\n\\ndef main() -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\\"--solution-csv\\\", type=Path, default=ROOT / \\\"data\\\" / \\\"splits\\\" / \\\"val_public_proxy.csv\\\")\\n    parser.add_argument(\\\"--questions-csv\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--baseline-predictions\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--candidate-predictions\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--baseline-adapter\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--candidate-adapter\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--base-model-path\\\", default=\\\"\\\")\\n    parser.add_argument(\\\"--seeds\\\", default=\\\"42\\\")\\n    parser.add_argument(\\\"--limit\\\", type=int, default=0)\\n    parser.add_argument(\\\"--family-regression-tolerance\\\", type=float, default=0.0)\\n    parser.add_argument(\\\"--min-net-gain\\\", type=float, default=0.0)\\n    parser.add_argument(\\\"--min-boxed-rate\\\", type=float, default=0.98)\\n    parser.add_argument(\\\"--output-dir\\\", type=Path, default=ROOT / \\\"artifacts\\\" / \\\"solve_rate_gate\\\")\\n    parser.add_argument(\\\"--json-output\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--self-test\\\", action=\\\"store_true\\\")\\n    args = parser.parse_args()\\n\\n    if args.self_test:\\n        return run_self_test(args.solution_csv, args.output_dir, args.limit)\\n\\n    solution = normalize_solution(args.solution_csv, args.limit)\\n    questions_csv = args.questions_csv or args.solution_csv\\n    questions = pd.read_csv(questions_csv)\\n    id_col = row_id_column(questions)\\n    if id_col != \\\"id\\\":\\n        questions = questions.rename(columns={id_col: \\\"id\\\"})\\n    questions[\\\"id\\\"] = questions[\\\"id\\\"].astype(str)\\n    if args.limit > 0:\\n        questions = questions[questions[\\\"id\\\"].isin(set(solution[\\\"id\\\"]))].copy()\\n\\n    csv_mode = args.baseline_predictions is not None and args.candidate_predictions is not None\\n    adapter_mode = args.baseline_adapter is not None and args.candidate_adapter is not None\\n    if csv_mode == adapter_mode:\\n        raise SystemExit(\\\"Choose exactly one mode: prediction CSVs or adapter paths.\\\")\\n\\n    if csv_mode:\\n        baseline = score_predictions(\\n            solution,\\n            predictions_from_csv(args.baseline_predictions, \\\"baseline\\\"),  # type: ignore[arg-type]\\n            \\\"baseline\\\",\\n        )\\n        candidate = score_predictions(\\n            solution,\\n            predictions_from_csv(args.candidate_predictions, \\\"candidate\\\"),  # type: ignore[arg-type]\\n            \\\"candidate\\\",\\n        )\\n    else:\\n        baseline, candidate = prediction_frames_from_adapters(\\n            solution,\\n            questions,\\n            baseline_adapter=args.baseline_adapter,  # type: ignore[arg-type]\\n            candidate_adapter=args.candidate_adapter,  # type: ignore[arg-type]\\n            base_model_path=resolve_base_model_path(args.base_model_path),\\n            seeds=parse_seeds(args.seeds),\\n        )\\n\\n    approved, reasons, comparison = compare(\\n        baseline,\\n        candidate,\\n        family_regression_tolerance=args.family_regression_tolerance,\\n        min_net_gain=args.min_net_gain,\\n        min_boxed_rate=args.min_boxed_rate,\\n    )\\n    args.output_dir.mkdir(parents=True, exist_ok=True)\\n    json_output = args.json_output or args.output_dir / \\\"solve_rate_gate_report.json\\\"\\n    report = {\\n        \\\"generated_at_utc\\\": utc_now(),\\n        \\\"status\\\": \\\"approve\\\" if approved else \\\"reject\\\",\\n        \\\"approved\\\": approved,\\n        \\\"reasons\\\": reasons,\\n        \\\"thresholds\\\": {\\n            \\\"family_regression_tolerance\\\": args.family_regression_tolerance,\\n            \\\"min_net_gain\\\": args.min_net_gain,\\n            \\\"min_boxed_rate\\\": args.min_boxed_rate,\\n        },\\n        \\\"inputs\\\": {\\n            \\\"solution_csv\\\": str(args.solution_csv),\\n            \\\"questions_csv\\\": str(questions_csv),\\n            \\\"baseline_predictions\\\": str(args.baseline_predictions) if args.baseline_predictions else \\\"\\\",\\n            \\\"candidate_predictions\\\": str(args.candidate_predictions) if args.candidate_predictions else \\\"\\\",\\n            \\\"baseline_adapter\\\": str(args.baseline_adapter) if args.baseline_adapter else \\\"\\\",\\n            \\\"candidate_adapter\\\": str(args.candidate_adapter) if args.candidate_adapter else \\\"\\\",\\n            \\\"seeds\\\": parse_seeds(args.seeds),\\n            \\\"limit\\\": args.limit,\\n        },\\n        \\\"comparison\\\": comparison,\\n    }\\n    json_output.write_text(json.dumps(report, indent=2), encoding=\\\"utf-8\\\")\\n    write_failures(args.output_dir / \\\"solve_rate_row_deltas.csv\\\", baseline, candidate)\\n    print(json.dumps(report, indent=2))\\n    return 0 if approved else 2\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    raise SystemExit(main())\\n\"\n}")
for rel, content in FILES.items():
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
    print('wrote', path, 'bytes=', path.stat().st_size)
for rel in ['src/competition_utils.py', 'scripts/evaluate_lora_adapter.py', 'scripts/solve_rate_gate.py']:
    py_compile.compile(str(ROOT / rel), doraise=True)
    print('compiled', rel)
print('=== V207A SCRIPT BOOTSTRAP END ===')


=== V207A SCRIPT BOOTSTRAP START ===
wrote /content/kg1/src/__init__.py bytes= 36
wrote /content/kg1/src/competition_utils.py bytes= 6850
wrote /content/kg1/scripts/evaluate_lora_adapter.py bytes= 14531
wrote /content/kg1/scripts/solve_rate_gate.py bytes= 15694
compiled src/competition_utils.py
compiled scripts/evaluate_lora_adapter.py
compiled scripts/solve_rate_gate.py
=== V207A SCRIPT BOOTSTRAP END ===


In [11]:
# CELL: download official train mirror and build seed-42 stratified validation CSV.
print('=== V207A VALIDATION DATA START ===')

import csv
import hashlib
import json
import pandas as pd
import random
import sys
from collections import defaultdict
from huggingface_hub import hf_hub_download

def sha256_path(path):
    h = hashlib.sha256()
    with pathlib.Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

if not TRAIN_CSV.exists():
    print('Downloading official train.csv mirror from HF dataset repo...')
    downloaded = hf_hub_download(
        repo_id='felipesp1983/kg1-nemotron-training',
        repo_type='dataset',
        filename='data/kaggle/unzipped/train.csv',
    )
    downloaded = pathlib.Path(downloaded)
    TRAIN_CSV.write_bytes(downloaded.read_bytes())
    print('downloaded_to =', downloaded)
else:
    print('TRAIN_CSV already exists:', TRAIN_CSV)

train_sha = sha256_path(TRAIN_CSV)
print('TRAIN_CSV sha256 =', train_sha)
if train_sha != TRAIN_CSV_SHA256:
    raise RuntimeError(f'train.csv SHA mismatch: {train_sha} != {TRAIN_CSV_SHA256}')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

utils_path = ROOT / 'src' / 'competition_utils.py'
print('competition_utils_path =', utils_path)
print('competition_utils_exists =', utils_path.exists())
if not utils_path.exists():
    raise FileNotFoundError(
        f'Missing {utils_path}. Run the "write embedded source files" cell before validation.'
    )

from src.competition_utils import classify_puzzle

train_df = pd.read_csv(TRAIN_CSV, dtype=str)
required_cols = {'id', 'prompt', 'answer'}
missing_cols = sorted(required_cols - set(train_df.columns))
if missing_cols:
    raise RuntimeError(f'train.csv missing required columns: {missing_cols}')

train_df['family'] = train_df['prompt'].map(classify_puzzle)

by_family = defaultdict(list)
for row in train_df.to_dict('records'):
    by_family[row['family']].append(row)

random.seed(42)
val_rows = []
for family, rows_in_family in sorted(by_family.items()):
    random.shuffle(rows_in_family)
    n_val = max(1, int(len(rows_in_family) * 0.10))
    print('family_split', family, 'total=', len(rows_in_family), 'val=', n_val)
    val_rows.extend(rows_in_family[:n_val])

random.shuffle(val_rows)
df = pd.DataFrame(val_rows)

missing = df[['prompt', 'answer']].isna().any(axis=1).sum()
if missing:
    raise RuntimeError(f'Validation rows missing prompt/answer: {missing}')

df.to_csv(VAL_CSV, index=False)

print('VAL_CSV rows =', len(df))
print('validation_source = official_train_seed42_stratified10_from_hf_mirror')
print('family_counts =')
print(df['family'].value_counts().sort_index().to_string())
print('VAL_CSV =', VAL_CSV)
print('=== V207A VALIDATION DATA END ===')


=== V207A VALIDATION DATA START ===
TRAIN_CSV already exists: /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_train.csv
TRAIN_CSV sha256 = d204af160633b638448723a437aa51c0db70fd0b64ff92f6ad6f52e5ac6377fa
competition_utils_path = /content/kg1/src/competition_utils.py
competition_utils_exists = True
family_split bit_manipulation total= 1602 val= 160
family_split equation_transform total= 1555 val= 155
family_split gravity_constant total= 1597 val= 159
family_split numeral_system total= 1576 val= 157
family_split text_encryption total= 1576 val= 157
family_split unit_conversion total= 1594 val= 159
VAL_CSV rows = 947
validation_source = official_train_seed42_stratified10_from_hf_mirror
family_counts =
family
bit_manipulation      160
equation_transform    155
gravity_constant      159
numeral_system        157
text_encryption       157
unit_conversion       159
VAL_CSV = /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_tra

In [12]:
# CELL: baseline adapter preflight.
print('=== V207A BASELINE PREFLIGHT START ===')
required = [BASELINE_ADAPTER / 'adapter_config.json', BASELINE_ADAPTER / 'adapter_model.safetensors']
for path in required:
    print('checking', path)
    if not path.exists():
        raise FileNotFoundError(f'Missing V194 baseline adapter file: {path}')
print('baseline_adapter_files =', sorted(p.name for p in BASELINE_ADAPTER.iterdir()))
print('=== V207A BASELINE PREFLIGHT END ===')


=== V207A BASELINE PREFLIGHT START ===
checking /content/drive/MyDrive/KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter/adapter_config.json
checking /content/drive/MyDrive/KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter/adapter_model.safetensors
baseline_adapter_files = ['adapter_config.json', 'adapter_model.safetensors']
=== V207A BASELINE PREFLIGHT END ===


In [13]:
# CELL: run V194 official-like ACC evaluation.
print('=== V207A V194 EVAL START ===')
eval_out = OUT_ROOT / 'v194_baseline_eval'
eval_log = REPORT_DIR / 'v194_baseline_eval.log'
cmd = [
    sys.executable,
    ROOT / 'scripts' / 'evaluate_lora_adapter.py',
    '--solution-csv', VAL_CSV,
    '--questions-csv', VAL_CSV,
    '--adapter', BASELINE_ADAPTER,
    '--base-model-path', MODEL_NAME,
    '--label', 'v194_baseline',
    '--seed', '42',
    '--limit', str(EVAL_LIMIT),
    '--output-dir', eval_out,
]
run_cmd(cmd, cwd=ROOT, log_path=eval_log)
print('eval_out =', eval_out)
print('eval_log =', eval_log)
print('=== V207A V194 EVAL END ===')


=== V207A V194 EVAL START ===
+ /usr/bin/python3 /content/kg1/scripts/evaluate_lora_adapter.py --solution-csv /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_train_seed42_stratified10_val.csv --questions-csv /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_train_seed42_stratified10_val.csv --adapter /content/drive/MyDrive/KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter --base-model-path nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 --label v194_baseline --seed 42 --limit 0 --output-dir /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v194_baseline_eval
KG1 official-like adapter evaluation
generated_at_utc = 2026-05-06T16:38:49.074874+00:00
base_model_path = nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
adapter_dir = /content/drive/MyDrive/KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter
rows = 947
seed = 42
config = {
  "dtype": "auto",
  "enable_chunked_prefill": true,
  "enable_prefix_caching": t

In [14]:
# CELL: baseline self-compare smoke gate.
print('=== V207A BASELINE SELF GATE START ===')
self_gate_out = OUT_ROOT / 'v194_baseline_self_gate'
pred_csv = OUT_ROOT / 'v194_baseline_eval' / 'v194_baseline_predictions.csv'
if not pred_csv.exists():
    raise FileNotFoundError(f'Missing baseline predictions: {pred_csv}')
gate_log = REPORT_DIR / 'v194_baseline_self_gate.log'
cmd = [
    sys.executable,
    ROOT / 'scripts' / 'solve_rate_gate.py',
    '--solution-csv', VAL_CSV,
    '--baseline-predictions', pred_csv,
    '--candidate-predictions', pred_csv,
    '--family-regression-tolerance', '0.0',
    '--min-net-gain', '-0.000001',
    '--min-boxed-rate', '0.0',
    '--output-dir', self_gate_out,
]
run_cmd(cmd, cwd=ROOT, log_path=gate_log)
print('self_gate_out =', self_gate_out)
print('gate_log =', gate_log)
print('=== V207A BASELINE SELF GATE END ===')


=== V207A BASELINE SELF GATE START ===
+ /usr/bin/python3 /content/kg1/scripts/solve_rate_gate.py --solution-csv /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_train_seed42_stratified10_val.csv --baseline-predictions /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v194_baseline_eval/v194_baseline_predictions.csv --candidate-predictions /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v194_baseline_eval/v194_baseline_predictions.csv --family-regression-tolerance 0.0 --min-net-gain -0.000001 --min-boxed-rate 0.0 --output-dir /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v194_baseline_self_gate
{
  "generated_at_utc": "2026-05-06T17:26:11.914976+00:00",
  "status": "approve",
  "approved": true,
  "reasons": [],
  "thresholds": {
    "family_regression_tolerance": 0.0,
    "min_net_gain": -1e-06,
    "min_boxed_rate": 0.0
  },
  "inputs": {
    "solution_csv": "/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207

In [15]:
# CELL: final V207A summary. This cell does not submit.
print('=== V207A FINAL SUMMARY START ===')
import json
summary = {
    'version': VERSION,
    'status': 'v194_acc_gate_completed',
    'val_csv': str(VAL_CSV),
    'v194_eval_dir': str(OUT_ROOT / 'v194_baseline_eval'),
    'v194_predictions_csv': str(OUT_ROOT / 'v194_baseline_eval' / 'v194_baseline_predictions.csv'),
    'v194_per_task_csv': str(OUT_ROOT / 'v194_baseline_eval' / 'v194_baseline_per_task.csv'),
    'v194_eval_report': str(OUT_ROOT / 'v194_baseline_eval' / 'v194_baseline_eval_report.json'),
    'self_gate_report': str(OUT_ROOT / 'v194_baseline_self_gate' / 'solve_rate_gate_report.json'),
    'allow_kaggle_submit': ALLOW_KAGGLE_SUBMIT,
}
summary_path = OUT_ROOT / 'V207A_FINAL_RUN_SUMMARY.json'
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
print(json.dumps(summary, indent=2, sort_keys=True))
print('summary_path =', summary_path)
print('NEXT: evaluate V206C/public adapters only after this baseline per_task.csv is reviewed.')
print('=== V207A FINAL SUMMARY END ===')


=== V207A FINAL SUMMARY START ===
{
  "allow_kaggle_submit": false,
  "self_gate_report": "/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v194_baseline_self_gate/solve_rate_gate_report.json",
  "status": "v194_acc_gate_completed",
  "v194_eval_dir": "/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v194_baseline_eval",
  "v194_eval_report": "/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v194_baseline_eval/v194_baseline_eval_report.json",
  "v194_per_task_csv": "/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v194_baseline_eval/v194_baseline_per_task.csv",
  "v194_predictions_csv": "/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v194_baseline_eval/v194_baseline_predictions.csv",
  "val_csv": "/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_train_seed42_stratified10_val.csv",
  "version": "V207A_H100_ACC_GATE_20260506"
}
summary_path = /content/drive/MyDrive/KG1_NVIDIA_V207A/output_

In [16]:
import pandas as pd
from pathlib import Path

root = Path('/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate')
per_task = pd.read_csv(root / 'v194_baseline_eval' / 'v194_baseline_per_task.csv')
print(per_task.to_string(index=False))

pred = pd.read_csv(root / 'v194_baseline_eval' / 'v194_baseline_predictions.csv')
print(pred.columns.tolist())


         task_type  total  correct  accuracy  truncated  truncation_rate
  bit_manipulation    160      135  0.843750          1         0.006250
equation_transform    155       55  0.354839          0         0.000000
  gravity_constant    159      159  1.000000          0         0.000000
    numeral_system    157      157  1.000000          0         0.000000
   text_encryption    157      157  1.000000          0         0.000000
   unit_conversion    159      159  1.000000          0         0.000000
           OVERALL    947      822  0.868004          1         0.001056
['id', 'prompt_x', 'answer', 'family', 'prompt_y', 'raw_output', 'prediction', 'prompt_tokens', 'completion_tokens', 'finish_reason', 'type', 'correct', 'truncated']


In [17]:
import pandas as pd
from pathlib import Path

root = Path('/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate')
pred = pd.read_csv(root / 'v194_baseline_eval' / 'v194_baseline_predictions.csv')

out = root / 'analysis_v194_errors'
out.mkdir(parents=True, exist_ok=True)

cols = ['id', 'family', 'type', 'correct', 'truncated', 'finish_reason',
        'answer', 'prediction', 'prompt_x', 'raw_output',
        'prompt_tokens', 'completion_tokens']

errors = pred[pred['correct'] == False].copy()
eq_errors = errors[errors['family'] == 'equation_transform'].copy()
bit_errors = errors[errors['family'] == 'bit_manipulation'].copy()

errors[cols].to_csv(out / 'all_errors.csv', index=False)
eq_errors[cols].to_csv(out / 'equation_transform_errors.csv', index=False)
bit_errors[cols].to_csv(out / 'bit_manipulation_errors.csv', index=False)

print('all_errors', len(errors), out / 'all_errors.csv')
print('equation_transform_errors', len(eq_errors), out / 'equation_transform_errors.csv')
print('bit_manipulation_errors', len(bit_errors), out / 'bit_manipulation_errors.csv')

print('\nfinish_reason by error family:')
print(errors.groupby(['family', 'finish_reason', 'truncated']).size().reset_index(name='n').to_string(index=False))

print('\nfirst equation_transform errors:')
for _, r in eq_errors.head(10).iterrows():
    print('\n---', r['id'], '---')
    print('answer:', r['answer'])
    print('prediction:', r['prediction'])
    print(str(r['prompt_x'])[:1200])


all_errors 125 /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/analysis_v194_errors/all_errors.csv
equation_transform_errors 100 /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/analysis_v194_errors/equation_transform_errors.csv
bit_manipulation_errors 25 /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/analysis_v194_errors/bit_manipulation_errors.csv

finish_reason by error family:
            family finish_reason  truncated   n
  bit_manipulation        length       True   1
  bit_manipulation          stop      False  24
equation_transform          stop      False 100

first equation_transform errors:

--- 09d5ee68 ---
answer: !?'!
prediction: |:>>
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
<:+|& = <:|&
'&+>? = '&>?
$!*!| = !!:&
|:-?' = -&$
Now, determine the result for: |:*>>

--- 00457d26 ---
answer: @&
prediction: [[!'
In Alice's Wonderland, a secret set of transformation r

In [18]:
from pathlib import Path

roots = [
    Path('/content/drive/MyDrive/KG1_NVIDIA_V206C/output_v206c_delta_scale/scaled_adapters'),
    Path('/content/drive/MyDrive/KG1_NVIDIA_V206B/output_v206b_answer_only_h100_loss_gated'),
    Path('/content/drive/MyDrive/KG1_NVIDIA_V202D'),
]

for root in roots:
    print('\nROOT:', root, 'exists=', root.exists())
    if root.exists():
        for cfg in sorted(root.rglob('adapter_config.json'))[:50]:
            adapter_dir = cfg.parent
            weights = adapter_dir / 'adapter_model.safetensors'
            print(' ', adapter_dir, 'weights=', weights.exists())



ROOT: /content/drive/MyDrive/KG1_NVIDIA_V206C/output_v206c_delta_scale/scaled_adapters exists= True
  /content/drive/MyDrive/KG1_NVIDIA_V206C/output_v206c_delta_scale/scaled_adapters/adapter_s0p000 weights= True
  /content/drive/MyDrive/KG1_NVIDIA_V206C/output_v206c_delta_scale/scaled_adapters/adapter_s0p010 weights= True
  /content/drive/MyDrive/KG1_NVIDIA_V206C/output_v206c_delta_scale/scaled_adapters/adapter_s0p020 weights= True
  /content/drive/MyDrive/KG1_NVIDIA_V206C/output_v206c_delta_scale/scaled_adapters/adapter_s0p050 weights= True
  /content/drive/MyDrive/KG1_NVIDIA_V206C/output_v206c_delta_scale/scaled_adapters/adapter_s0p100 weights= True

ROOT: /content/drive/MyDrive/KG1_NVIDIA_V206B/output_v206b_answer_only_h100_loss_gated exists= True
  /content/drive/MyDrive/KG1_NVIDIA_V206B/output_v206b_answer_only_h100_loss_gated/train_v206b_answer_only_1s_lr1e9/final_adapter weights= True

ROOT: /content/drive/MyDrive/KG1_NVIDIA_V202D exists= True
  /content/drive/MyDrive/KG1_NVIDI

In [22]:
import subprocess
from pathlib import Path

def stream_process(cmd, cwd=None, env=None, log_path=None):
    cmd = [str(part) for part in cmd]
    print('+', ' '.join(cmd))
    log_handle = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_handle = log_path.open('w', encoding='utf-8')

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        if log_handle:
            log_handle.write(line)

    rc = proc.wait()
    if log_handle:
        log_handle.close()
    print('returncode =', rc)
    return rc

print('stream_process ready')


stream_process ready


In [ ]:
from pathlib import Path

ROOT = Path('/content/kg1')
SCRIPT_DIR = ROOT / 'scripts'
EVAL_SCRIPT = SCRIPT_DIR / 'evaluate_lora_adapter.py'
GATE_SCRIPT = SCRIPT_DIR / 'solve_rate_gate.py'

VAL_CSV = Path('/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_train_seed42_stratified10_val.csv')
OUT_ROOT = Path('/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate')
REPORT_DIR = OUT_ROOT / 'reports'
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

candidate = Path('/content/drive/MyDrive/KG1_NVIDIA_V206C/output_v206c_delta_scale/scaled_adapters/adapter_s0p010')
label = 'v206c_s0p010'

out = OUT_ROOT / f'{label}_eval'
log = REPORT_DIR / f'{label}_eval.log'

print('ROOT =', ROOT, ROOT.exists())
print('EVAL_SCRIPT =', EVAL_SCRIPT, EVAL_SCRIPT.exists())
print('GATE_SCRIPT =', GATE_SCRIPT, GATE_SCRIPT.exists())
print('VAL_CSV =', VAL_CSV, VAL_CSV.exists())
print('candidate =', candidate, candidate.exists())
print('out =', out)
print('log =', log)

if not EVAL_SCRIPT.exists():
    raise FileNotFoundError(EVAL_SCRIPT)
if not VAL_CSV.exists():
    raise FileNotFoundError(VAL_CSV)
if not (candidate / 'adapter_model.safetensors').exists():
    raise FileNotFoundError(candidate / 'adapter_model.safetensors')

cmd = [
    '/usr/bin/python3', str(EVAL_SCRIPT),
    '--solution-csv', str(VAL_CSV),
    '--questions-csv', str(VAL_CSV),
    '--adapter', str(candidate),
    '--base-model-path', MODEL_NAME,
    '--label', label,
    '--seed', '42',
    '--limit', '0',
    '--output-dir', str(out),
]

rc = stream_process(cmd, cwd=ROOT, log_path=log)
if rc != 0:
    raise RuntimeError(f'{label} eval failed: {rc}')

print('candidate_predictions =', out / f'{label}_predictions.csv')
print('candidate_per_task =', out / f'{label}_per_task.csv')


ROOT = /content/kg1 True
EVAL_SCRIPT = /content/kg1/scripts/evaluate_lora_adapter.py True
GATE_SCRIPT = /content/kg1/scripts/solve_rate_gate.py True
VAL_CSV = /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_train_seed42_stratified10_val.csv True
candidate = /content/drive/MyDrive/KG1_NVIDIA_V206C/output_v206c_delta_scale/scaled_adapters/adapter_s0p010 True
out = /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v206c_s0p010_eval
log = /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/reports/v206c_s0p010_eval.log
+ /usr/bin/python3 /content/kg1/scripts/evaluate_lora_adapter.py --solution-csv /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_train_seed42_stratified10_val.csv --questions-csv /content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/validation/official_train_seed42_stratified10_val.csv --adapter /content/drive/MyDrive/KG1_NVIDIA_V206C/output_v206c_delta_scale/scaled_adapters/adap